# AIQToolkit NVIDIA Blueprint Launcher

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/plturrell/aiqtoolkit-nvidia/blob/main/launch.ipynb)

This notebook allows you to launch and interact with the AIQToolkit NVIDIA Blueprint from any environment that supports Jupyter notebooks.

## Prerequisites

- Python 3.8+
- NVIDIA API Key (optional)
- GPU with CUDA support (optional but recommended)

## Setup Environment

First, let's check if we're running in a compatible environment and install required dependencies.

In [ ]:
# Install requirements
!pip install -q requests python-dotenv flask pillow torch

# Import necessary libraries
import os
import sys
import subprocess
import json
import time
import IPython
from IPython.display import display, HTML, clear_output
import threading
import requests
from pathlib import Path

## Check for NVIDIA GPU

Let's verify if a NVIDIA GPU is available. The blueprint will work without a GPU, but having one will enable better performance.

In [ ]:
def check_nvidia_gpu():
    """Check if NVIDIA GPU is available"""
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
        if result.returncode == 0:
            print("✅ NVIDIA GPU detected!")
            # Extract some basic info
            print("\nGPU Information:")
            print("----------------")
            for line in result.stdout.split('\n'):
                if "NVIDIA" in line and "|" in line and ("%" in line or "MiB" in line):
                    print(line)
            return True
        else:
            print("⚠️ No NVIDIA GPU detected. The blueprint will run in CPU-only mode.")
            return False
    except FileNotFoundError:
        print("⚠️ nvidia-smi not found. No NVIDIA GPU detected or drivers not installed.")
        print("The blueprint will run in CPU-only mode.")
        return False

# Check for GPU
has_gpu = check_nvidia_gpu()

## Configure NVIDIA API Key

To use NVIDIA services, you can provide an API key. This is optional but enables additional functionality.

In [ ]:
# Configure NVIDIA API Key
from IPython.display import display
import ipywidgets as widgets

# Create input widget for API key
api_key_input = widgets.Password(
    description='NVIDIA API Key:',
    placeholder='Enter your NVIDIA API key (optional)',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

# Create input widget for endpoint
endpoint_input = widgets.Text(
    value='https://jupyter0-s1ondnjfx.brevlab.com',
    description='NVIDIA Endpoint:',
    placeholder='Enter NVIDIA endpoint URL',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

# Create a checkbox for saving credentials
save_checkbox = widgets.Checkbox(
    value=True,
    description='Save configuration for this session',
    style={'description_width': 'initial'}
)

# Display the widgets
display(api_key_input)
display(endpoint_input)
display(save_checkbox)

# Create a button to confirm
button = widgets.Button(
    description='Save Configuration',
    button_style='primary',
    tooltip='Click to save NVIDIA configuration'
)

output = widgets.Output()

def on_button_clicked(b):
    with output:
        clear_output()
        nvidia_api_key = api_key_input.value.strip()
        nvidia_endpoint = endpoint_input.value.strip()
        
        if nvidia_api_key:
            print(f"✅ NVIDIA API Key configured: {nvidia_api_key[:5]}...")
            os.environ['NVIDIA_API_KEY'] = nvidia_api_key
        else:
            print("ℹ️ No NVIDIA API Key provided. Some features may be limited.")
            
        if nvidia_endpoint:
            print(f"✅ NVIDIA Endpoint configured: {nvidia_endpoint}")
            os.environ['NVIDIA_ENDPOINT'] = nvidia_endpoint
        else:
            print("ℹ️ No NVIDIA Endpoint provided. Using default endpoint.")
        
        if save_checkbox.value:
            # Save to .env file in the current directory
            with open('.env', 'w') as f:
                if nvidia_api_key:
                    f.write(f"NVIDIA_API_KEY={nvidia_api_key}\n")
                if nvidia_endpoint:
                    f.write(f"NVIDIA_ENDPOINT={nvidia_endpoint}\n")
            print("✅ Configuration saved to .env file")
            
        print("\nReady to launch the blueprint!")

button.on_click(on_button_clicked)
display(button)
display(output)

## Clone Repository

Let's clone the AIQToolkit NVIDIA Blueprint repository to get the latest version.

In [ ]:
# Clone the repository if it doesn't exist
def clone_repo():
    repo_dir = Path("./aiqtoolkit-nvidia")
    
    if repo_dir.exists():
        print(f"Repository already exists at {repo_dir}")
        # Pull latest changes
        result = subprocess.run(
            ["git", "pull"], 
            cwd=repo_dir,
            capture_output=True, 
            text=True
        )
        print(result.stdout)
        return repo_dir
    
    print("Cloning AIQToolkit NVIDIA Blueprint repository...")
    result = subprocess.run(
        ["git", "clone", "https://github.com/plturrell/aiqtoolkit-nvidia.git"], 
        capture_output=True, 
        text=True
    )
    
    if result.returncode == 0:
        print("✅ Repository cloned successfully")
    else:
        print(f"❌ Error cloning repository: {result.stderr}")
    
    return repo_dir

repo_dir = clone_repo()

## Launch Web UI and API Servers

Now, let's start the web UI and API servers to access the NVIDIA Blueprint.

In [ ]:
# Servers for the blueprint
api_server = None
ui_server = None

# Start the servers
def start_servers(api_port=8000, ui_port=8080):
    global api_server, ui_server
    
    # Stop existing servers if running
    if api_server or ui_server:
        stop_servers()
    
    web_ui_dir = repo_dir / "web-ui"
    
    # Start API server using Python's built-in HTTP server
    print(f"🚀 Starting API server on port {api_port}...")
    api_server = subprocess.Popen(
        [sys.executable, "-m", "http.server", str(api_port), "--directory", str(web_ui_dir / "api")],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        env=dict(os.environ)
    )
    
    # Start UI server using Python's built-in HTTP server
    print(f"🚀 Starting UI server on port {ui_port}...")
    ui_server = subprocess.Popen(
        [sys.executable, "-m", "http.server", str(ui_port), "--directory", str(web_ui_dir / "public")],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        env=dict(os.environ)
    )
    
    # Wait for servers to start
    time.sleep(2)
    
    # Check if servers are running
    api_running = api_server.poll() is None
    ui_running = ui_server.poll() is None
    
    if api_running:
        print(f"✅ API server running at http://localhost:{api_port}")
    else:
        print("❌ API server failed to start")
        
    if ui_running:
        print(f"✅ UI server running at http://localhost:{ui_port}")
    else:
        print("❌ UI server failed to start")
        
    return api_running, ui_running

# Stop the servers
def stop_servers():
    global api_server, ui_server
    
    print("Stopping servers...")
    
    if api_server:
        api_server.terminate()
        api_server = None
        
    if ui_server:
        ui_server.terminate()
        ui_server = None
    
    print("✅ Servers stopped")

# Launch the servers
api_port = 8000
ui_port = 8080
api_running, ui_running = start_servers(api_port, ui_port)

## Open Control Tower UI

Now that the servers are running, let's open the NVIDIA Control Tower interface.

In [ ]:
# Create a clickable button to open the Control Tower
if ui_running:
    control_tower_url = f"http://localhost:{ui_port}/control-tower.html"
    
    html_button = f"""
    <div style="text-align: center; margin: 20px 0;">
        <a href="{control_tower_url}" target="_blank" 
           style="display: inline-block; background-color: #76B900; color: white; padding: 15px 30px; 
                  text-decoration: none; font-weight: bold; border-radius: 5px; font-size: 16px;">
            Open NVIDIA Control Tower
        </a>
    </div>
    
    <div style="text-align: center;">
        <p>Click the button above to open the NVIDIA Control Tower interface in a new tab.</p>
        <p>From there, you can monitor and interact with NVIDIA services.</p>
    </div>
    """
    
    display(HTML(html_button))
else:
    print("⚠️ UI server is not running. Cannot open Control Tower.")

## Open Blueprint Console

Alternatively, you can use the Blueprint Console for a command-line style interface.

In [ ]:
# Create a clickable button to open the Blueprint Console
if ui_running:
    console_url = f"http://localhost:{ui_port}/blueprint-console.html"
    
    html_button = f"""
    <div style="text-align: center; margin: 20px 0;">
        <a href="{console_url}" target="_blank" 
           style="display: inline-block; background-color: #333; color: white; padding: 15px 30px; 
                  text-decoration: none; font-weight: bold; border-radius: 5px; font-size: 16px;">
            Open Blueprint Console
        </a>
    </div>
    
    <div style="text-align: center;">
        <p>Click the button above to open the Blueprint Console interface in a new tab.</p>
        <p>This provides a command-line style interface to control the blueprint.</p>
    </div>
    """
    
    display(HTML(html_button))
else:
    print("⚠️ UI server is not running. Cannot open Blueprint Console.")

## Test NVIDIA Services

Let's run a quick test of the NVIDIA services.

In [ ]:
# Test NVIDIA services
def test_nvidia_services():
    if not api_running:
        print("⚠️ API server is not running. Cannot test NVIDIA services.")
        return
    
    print("Testing NVIDIA services...")
    
    try:
        response = requests.get(f"http://localhost:{api_port}/health")
        if response.status_code == 200:
            health_data = response.json()
            print(f"✅ API server health check: {health_data['status']}")
        else:
            print(f"❌ API server health check failed: {response.status_code}")
            
        # Test each NVIDIA service
        services = ["nim", "api", "nemo", "triton"]
        
        results = []
        for service in services:
            try:
                print(f"Testing {service.upper()}...")
                response = requests.post(
                    f"http://localhost:{api_port}/nvidia",
                    json={
                        "prompt": "Hello, this is a test message.",
                        "service_type": service,
                        "model": service == "nemo" and "nemo-model" or "llama-3.1-70b-instruct"
                    }
                )
                
                if response.status_code == 200:
                    data = response.json()
                    status = data.get('status', 'unknown')
                    
                    if status == 'success':
                        print(f"✅ {service.upper()} test successful")
                        results.append((service, "success"))
                    else:
                        print(f"❌ {service.upper()} test failed: {status}")
                        results.append((service, "error"))
                else:
                    print(f"❌ {service.upper()} test failed: {response.status_code}")
                    results.append((service, "error"))
                    
            except Exception as e:
                print(f"❌ {service.upper()} test error: {str(e)}")
                results.append((service, "error"))
        
        # Summarize results
        success_count = sum(1 for _, status in results if status == "success")
        print(f"\nTest summary: {success_count}/{len(services)} services tested successfully")
        
    except Exception as e:
        print(f"❌ Error testing NVIDIA services: {str(e)}")

# Run the test
test_nvidia_services()

## Interactive Command Console

Let's create an interactive console to run commands directly from the notebook.

In [ ]:
from ipywidgets import Text, Button, Output, VBox, HBox, HTML
from IPython.display import display, clear_output

# Create widgets
command_input = Text(
    description='Command:',
    placeholder='Type command (e.g. status, test, help)',
    layout={'width': '400px'}
)

run_button = Button(
    description='Run',
    button_style='success',
    tooltip='Run the command'
)

console_output = Output(
    layout={'width': '100%', 'height': '300px', 'border': '1px solid #ddd', 'overflow': 'auto'}
)

# Define command handling
def handle_command(command):
    with console_output:
        print(f"\033[94m> {command}\033[0m")
        
        if command.lower() == 'help':
            print("\nAvailable commands:")
            print("  status    - Show service status")
            print("  test      - Test NVIDIA services")
            print("  restart   - Restart services")
            print("  nvidia-smi - Show GPU information")
            print("  clear     - Clear console output")
            print("  help      - Show this help message")
        
        elif command.lower() == 'status':
            print("\nService Status:")
            print(f"  API Server: {'Running' if api_running else 'Stopped'} on port {api_port}")
            print(f"  UI Server: {'Running' if ui_running else 'Stopped'} on port {ui_port}")
            print(f"  NVIDIA API Key: {'Configured' if os.environ.get('NVIDIA_API_KEY') else 'Not configured'}")
            print(f"  NVIDIA Endpoint: {os.environ.get('NVIDIA_ENDPOINT', 'Not configured')}")
        
        elif command.lower() == 'test':
            test_nvidia_services()
        
        elif command.lower() == 'restart':
            print("Restarting services...")
            api_running, ui_running = start_servers(api_port, ui_port)
        
        elif command.lower() == 'nvidia-smi':
            try:
                result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
                if result.returncode == 0:
                    print(result.stdout)
                else:
                    print("❌ Error running nvidia-smi")
            except Exception as e:
                print(f"❌ Error: {str(e)}")
        
        elif command.lower() == 'clear':
            clear_output()
            print("Console cleared")
        
        else:
            print(f"Unknown command: {command}")
            print("Type 'help' for available commands")

# Handle button click
def on_run_button_clicked(b):
    command = command_input.value
    if command.strip():
        handle_command(command)
        command_input.value = ""  # Clear input after running

# Handle Enter key in input field
def on_input_submit(sender):
    handle_command(command_input.value)
    command_input.value = ""  # Clear input after running

# Connect event handlers
run_button.on_click(on_run_button_clicked)
command_input.on_submit(on_input_submit)

# Display the console
console_title = HTML("<h3>Blueprint Interactive Console</h3>")
console = VBox([console_title, HBox([command_input, run_button]), console_output])
display(console)

# Show initial help message
with console_output:
    print("AIQToolkit NVIDIA Blueprint Console")
    print("Type 'help' for available commands")
    print("")

## Cleanup Resources

When you're done using the blueprint, run this cell to stop all servers and release resources.

In [ ]:
# Stop the servers and clean up
def cleanup():
    stop_servers()
    print("Resources released. You can safely close this notebook.")

# Register cleanup to run when kernel is shut down
import atexit
atexit.register(cleanup)